# M10 Lab — ML Bridge: Pipelines, CV, Tuning

**Dataset:** `housing_prices.csv` &nbsp;|&nbsp; **Anchor:** Géron Ch2 p.40–115

Some cells are marked `# [AI-OFF]` — those must be completed without Gemma's help. Log every Gemma interaction in `AI_USE.md`.

## L10.1 — Frame the problem `[Géron Ch2 p.40–55]`

Before any code, write the four-line framing as a comment block.

In [ ]:
# OBJECTIVE: predict median_house_value to support pricing decisions
# METRIC:    RMSE in USD
# BASELINE:  DummyRegressor(strategy='mean')
# CONSTRAINT: inference < 100ms; need feature importances for explainability

import numpy as np
import pandas as pd
np.random.seed(42)

df = pd.read_csv('housing_prices.csv')
df.head()

## L10.2 — Stratified train/test split `[Géron Ch2 p.55–65]`

In [ ]:
from sklearn.model_selection import train_test_split

df['income_cat'] = pd.cut(df['median_income'],
                          bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
                          labels=[1, 2, 3, 4, 5])

train, test = train_test_split(df, test_size=0.2,
                               stratify=df['income_cat'],
                               random_state=42)

for s in (train, test):
    s.drop(columns='income_cat', inplace=True)

y_train = train.pop('median_house_value')
y_test  = test.pop('median_house_value')
X_train, X_test = train, test
print(X_train.shape, X_test.shape)

## L10.3 — Choose a scaler with rationale `[Géron Ch2 p.83–88]`

In the markdown cell below, write 1 sentence per numeric feature explaining your scaler choice.

In [ ]:
X_train.describe()

*(your rationale here — replace this text)*
- `median_income` — 
- `housing_median_age` — 
- `total_rooms` — 

## L10.4 — Pipeline + ColumnTransformer assembly &nbsp;⚠️ **[AI-OFF]** `[Géron Ch2 p.85–95]`

Hand-assemble the preprocessing pipeline below **without Gemma's help**. After the code, write a markdown cell listing **at least 2 leakage paths** that would open up if these transformers were fit before the train/test split.

In [ ]:
# [AI-OFF]
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

numeric_features = ['median_income', 'housing_median_age', 'total_rooms',
                    'total_bedrooms', 'population', 'households']
categorical_features = ['ocean_proximity']

# TODO: build numeric_pipeline (imputer -> scaler)
numeric_pipeline = Pipeline([
    # ('imputer', ...),
    # ('scaler',  ...),
])

# TODO: build categorical_pipeline (imputer -> encoder)
categorical_pipeline = Pipeline([
    # ('imputer', ...),
    # ('encoder', ...),
])

preprocessor = ColumnTransformer([
    # ('num', numeric_pipeline, numeric_features),
    # ('cat', categorical_pipeline, categorical_features),
])

full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(random_state=42)),
])

full_pipeline

**[AI-OFF] leakage analysis — write your own answer:**

1. *(leakage path 1)*
2. *(leakage path 2)*
3. *(optional: a third path)*

## L10.5 — Custom transformer `[Géron Ch2 p.90–95]`

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class RoomsPerHousehold(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rooms = X[:, 0] / X[:, 1]
        return np.c_[X, rooms]

## L10.6 — Cross-validation `[Géron Ch2 p.95–100]`

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

full_pipeline.fit(X_train, y_train)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(full_pipeline, X_train, y_train,
                         scoring='neg_root_mean_squared_error', cv=cv)
print(f'CV RMSE: {-scores.mean():.0f} ± {scores.std():.0f}')

## L10.7 — Hyperparameter tuning `[Géron Ch2 p.100–110]`

**Keep the grid small.** Larger grids overfit the validation folds.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth':    [None, 20],
}

search = GridSearchCV(full_pipeline, param_grid, cv=3,
                      scoring='neg_root_mean_squared_error',
                      n_jobs=-1, refit=True)
search.fit(X_train, y_train)
print(search.best_params_, -search.best_score_)

## L10.8 — Persistence `[Géron Ch2 p.110–115]`

In [ ]:
import joblib

joblib.dump(search.best_estimator_, 'pipeline.joblib')

# Reload and verify the predictions match (round-trip test)
pipe = joblib.load('pipeline.joblib')
preds_orig   = search.best_estimator_.predict(X_test.head(5))
preds_loaded = pipe.predict(X_test.head(5))
assert np.allclose(preds_orig, preds_loaded), 'reload mismatch'
print('Round-trip OK. Final test RMSE:', np.sqrt(((pipe.predict(X_test) - y_test) ** 2).mean()))

---
## Submission checklist

- [ ] All cells run top-to-bottom from a fresh kernel
- [ ] L10.4 [AI-OFF] cell completed with at least 2 documented leakage paths
- [ ] `pipeline.joblib` artifact exists alongside this notebook
- [ ] `AI_USE.md` lists every Gemma prompt, response, and your accept/edit/reject decision
- [ ] Final test RMSE printed in the last cell